# spaCy: обработка естественного языка для научных текстов

**Проект «ИИ для учёных».** Практический блокнот к разделу «Инструменты».

## Что делает этот инструмент

spaCy — промышленная библиотека обработки естественного языка (NLP) для Python с поддержкой более 75 языков, включая русский. Она выполняет базовые и среднего уровня задачи анализа текста: токенизацию, лемматизацию, разметку частей речи, разбор синтаксических зависимостей и выделение именованных сущностей (NER).

Для исследователя это значит: любой корпус текстов — статьи, отчёты, патенты, клинические записи — можно быстро преобразовать в структурированные лингвистические данные без написания правил вручную. spaCy часто выступает первым шагом пайплайна перед классификаторами, тематическими моделями или языковыми моделями: она снижает размерность словаря, нормализует словоформы и извлекает сущности, которые затем становятся признаками для ML.

В этом блокноте мы разберём полный цикл: сырой русский текст → токенизация и лемматизация → частеречная разметка и синтаксис → именованные сущности → статистический анализ → препроцессинг для ML. Расчётное время прохождения — около пяти минут.

## Ключевые понятия

- **Токенизация** — разбиение текста на минимальные единицы (токены): слова, знаки препинания, числа. spaCy делает это с учётом языковых правил, а не просто по пробелам.
- **Лемматизация** — приведение словоформы к словарной форме (лемме): «исследований» → «исследование», «изучали» → «изучать». Уменьшает разнообразие словаря и улучшает сравнение текстов.
- **POS-разметка** (Part-of-Speech) — определение части речи каждого токена: существительное, глагол, прилагательное и т.д. Используется для фильтрации текста или построения признаков.
- **Синтаксические зависимости** (Dependency Parsing) — дерево связей между словами предложения: какое слово является подлежащим, дополнением, определением. Позволяет извлекать структурированные факты из текста.
- **Именованные сущности** (NER, Named Entity Recognition) — автоматическое распознавание в тексте объектов реального мира: организаций, персон, географических названий, дат, числовых значений. Критически важно для извлечения информации из научных публикаций.
- **Пайплайн** — последовательность компонентов обработки текста в spaCy (tokenizer → tagger → parser → ner). Каждый компонент добавляет аннотации к объекту документа `Doc`.

## 1. Установка библиотек

In [ ]:
# Устанавливаем spaCy и скачиваем языковые модели.
# ru_core_news_sm — малая модель для русского языка (~13 МБ):
#   токенизация, POS, синтаксис, NER, обучена на новостных текстах.
# en_core_web_sm — малая модель для английского (~12 МБ), нужна для
#   демонстрации совместной работы с двуязычными корпусами.
# Время установки в Colab: около 1–2 минут.

!pip install -q spacy

# Загружаем модели напрямую через pip — наиболее надёжный способ в Colab,
# не зависящий от прав sudo и стабильности spacy download.
!pip install -q \
    https://github.com/explosion/spacy-models/releases/download/ru_core_news_sm-3.7.0/ru_core_news_sm-3.7.0-py3-none-any.whl \
    https://github.com/explosion/spacy-models/releases/download/en_core_web_sm-3.7.0/en_core_web_sm-3.7.0-py3-none-any.whl

import spacy
print(f"spaCy версия: {spacy.__version__}")

## 2. Единый стиль графиков

Графики оформляются в едином визуальном стиле сайта «ИИ для учёных». Ниже встроено содержимое стилевого шаблона `notebooks/viz_style.py`; вызов `apply_viz_style()` настраивает matplotlib один раз на весь блокнот.

In [ ]:
# Единый стилевой шаблон графиков проекта «ИИ для учёных».
# Значения синхронизированы с токенами темы сайта (styles/tokens.css).
VIZ_TOKENS = {
    "background": "#ffffff",
    "node_fill":  "#eef1f6",
    "node_text":  "#0e1726",
    "edge":       "#4d5e78",
    "grid":       "#dce2ec",
    "series":     ["#2563eb", "#0d9488", "#b45309", "#4d5e78"],
}
VIZ = VIZ_TOKENS


def apply_viz_style():
    """Настраивает matplotlib под единый стиль сайта."""
    import matplotlib as mpl
    from cycler import cycler
    t = VIZ_TOKENS
    mpl.rcParams.update({
        "font.family": "sans-serif",
        "font.sans-serif": ["Segoe UI", "DejaVu Sans", "Arial", "Helvetica"],
        "font.size": 12, "axes.titlesize": 15, "axes.titleweight": "semibold",
        "axes.labelsize": 13, "xtick.labelsize": 11, "ytick.labelsize": 11,
        "legend.fontsize": 11,
        "figure.facecolor": t["background"], "axes.facecolor": t["background"],
        "savefig.facecolor": t["background"], "figure.dpi": 110, "savefig.dpi": 150,
        "axes.prop_cycle": cycler(color=t["series"]),
        "text.color": t["node_text"], "axes.labelcolor": t["node_text"],
        "axes.titlecolor": t["node_text"], "axes.edgecolor": t["edge"],
        "xtick.color": t["edge"], "ytick.color": t["edge"], "axes.linewidth": 1.2,
        "axes.grid": True, "grid.color": t["grid"], "grid.linewidth": 1.0,
        "grid.linestyle": "-", "axes.axisbelow": True,
        "lines.linewidth": 2.0, "lines.markersize": 6, "patch.linewidth": 1.5,
        "axes.spines.top": False, "axes.spines.right": False,
        "legend.frameon": False,
    })


def get_palette(n=None):
    """Возвращает список цветов рядов из токенов темы."""
    series = VIZ_TOKENS["series"]
    if n is None:
        return list(series)
    return [series[i % len(series)] for i in range(n)]


apply_viz_style()
print("Единый стиль графиков подключён.")

## 3. Демонстрационные данные

Используем небольшой корпус из десяти предложений научно-популярной тематики на русском языке. Тексты охватывают несколько предметных областей, содержат именованные сущности (организации, персоны, географические объекты, даты) и разнообразные грамматические конструкции — это позволяет показать все возможности spaCy на одном наборе данных.

In [ ]:
# Демонстрационный корпус: десять предложений научной тематики.
# Каждое предложение содержит именованные сущности и разнообразную грамматику,
# чтобы показать полный набор возможностей spaCy.

CORPUS = [
    "Институт молекулярной биологии РАН опубликовал результаты исследования "
    "структуры белков теплового шока в 2023 году.",

    "Академик Иван Петров из Новосибирского государственного университета "
    "предложил новый метод синтеза графеновых нанотрубок.",

    "Группа учёных из MIT и Сколтеха провела серию экспериментов по изучению "
    "квантовых свойств двумерных материалов при температуре 4 Кельвина.",

    "Российский научный фонд выделил грант в размере 50 миллионов рублей "
    "на разработку вакцины против туберкулёза.",

    "Лаборатория нейросетевых технологий МГУ разрабатывает алгоритмы "
    "машинного обучения для анализа геномных последовательностей.",

    "В ходе экспедиции в Арктику в августе 2022 года были обнаружены "
    "залежи метангидратов на глубине 200 метров.",

    "Профессор Мария Соколова опубликовала монографию о роли микробиома "
    "в развитии аутоиммунных заболеваний.",

    "Европейский центр ядерных исследований ЦЕРН зафиксировал новые данные "
    "об аномальном магнитном моменте мюона в эксперименте CMS.",

    "Биохимики из Гарвардского университета синтезировали аналог пенициллина, "
    "устойчивый к действию бета-лактамаз.",

    "Согласно отчёту ВОЗ за 2021 год, смертность от онкологических заболеваний "
    "в странах с низким доходом снизилась на 12 процентов.",
]

# Объединяем в один документ для сводного анализа
FULL_TEXT = " ".join(CORPUS)

print(f"Корпус: {len(CORPUS)} предложений, {len(FULL_TEXT.split())} слов")
print("\nПример (первое предложение):")
print(CORPUS[0])

## 4. Применение инструмента

### Шаг 4.1. Загрузка модели и базовая токенизация

Загружаем русскоязычную модель `ru_core_news_sm`. Обработка текста в spaCy устроена просто: передаём строку в вызываемую модель `nlp(text)` и получаем объект `Doc`, аннотированный всеми компонентами пайплайна одновременно.

In [ ]:
import spacy

# Загружаем малую русскую модель.
# Если выше использовался pip install URL, модель уже установлена.
nlp_ru = spacy.load("ru_core_news_sm")

print("Компоненты пайплайна:", nlp_ru.pipe_names)
print()

# Обрабатываем первое предложение корпуса
sentence = CORPUS[0]
doc = nlp_ru(sentence)

print(f"Текст: «{sentence}»")
print(f"\nТокенов найдено: {len(doc)}")
print()

# Выводим таблицу: токен, лемма, часть речи, тег
print(f"{'Токен':<25} {'Лемма':<25} {'POS':<10} {'Теги'}")
print("-" * 75)
for token in doc:
    print(f"{token.text:<25} {token.lemma_:<25} {token.pos_:<10} {token.tag_}")

### Шаг 4.2. Лемматизация и фильтрация значимых слов

На практике для анализа текста нужны не все токены. Стоп-слова (предлоги, союзы, частицы), знаки препинания и числа обычно исключаются. Остаются значимые токены, приведённые к леммам, — именно они несут смысловую нагрузку.

In [ ]:
def extract_lemmas(text, nlp, pos_filter=None):
    """
    Возвращает список лемм значимых токенов.

    Параметры
    ----------
    text       : str  — исходный текст
    nlp        : spaCy Language — загруженная модель
    pos_filter : list[str] | None — оставить только указанные части речи
                 (например, ['NOUN', 'VERB', 'ADJ']); None = все значимые токены

    Значимый токен: не пунктуация, не стоп-слово, не пробел.
    """
    doc = nlp(text)
    lemmas = []
    for token in doc:
        if token.is_punct or token.is_space or token.is_stop:
            continue
        if pos_filter and token.pos_ not in pos_filter:
            continue
        lemmas.append(token.lemma_.lower())
    return lemmas


# Применяем к первому предложению: только существительные и прилагательные
nouns_adj = extract_lemmas(sentence, nlp_ru, pos_filter=["NOUN", "ADJ", "PROPN"])
print("Существительные и прилагательные (леммы):")
print(nouns_adj)

print()

# Применяем ко всему корпусу: все значимые токены
all_lemmas = []
for sent in CORPUS:
    all_lemmas.extend(extract_lemmas(sent, nlp_ru))

print(f"Всего значимых лемм в корпусе: {len(all_lemmas)}")
print(f"Уникальных лемм: {len(set(all_lemmas))}")

# Топ-15 самых частых лемм
from collections import Counter
top_lemmas = Counter(all_lemmas).most_common(15)
print("\nТоп-15 лемм по частоте:")
for lemma, count in top_lemmas:
    print(f"  {lemma:<25} {count}")

### Шаг 4.3. Частеречная разметка (POS) по всему корпусу

Посмотрим, как распределены части речи в нашем корпусе. Для научных текстов характерно преобладание существительных и прилагательных; высокая доля глаголов может указывать на описание процедур и методов.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from collections import Counter

# Собираем POS-статистику по всему корпусу
pos_counts = Counter()
for sent in CORPUS:
    doc = nlp_ru(sent)
    for token in doc:
        if not token.is_punct and not token.is_space:
            pos_counts[token.pos_] += 1

# Читаемые русские названия частей речи
POS_LABELS = {
    "NOUN":  "Существительное",
    "VERB":  "Глагол",
    "ADJ":   "Прилагательное",
    "PROPN": "Имя собственное",
    "ADV":   "Наречие",
    "ADP":   "Предлог",
    "CCONJ": "Сочинит. союз",
    "SCONJ": "Подчинит. союз",
    "DET":   "Артикль/детерм.",
    "NUM":   "Числительное",
    "PRON":  "Местоимение",
    "PART":  "Частица",
    "PUNCT": "Знак препинания",
    "X":     "Прочее",
}

# Отбираем топ-10 тегов для графика
top_pos = pos_counts.most_common(10)
labels  = [POS_LABELS.get(pos, pos) for pos, _ in top_pos]
values  = [count for _, count in top_pos]
palette = get_palette(len(values))

fig, ax = plt.subplots(figsize=(10, 5.5))
bars = ax.barh(labels[::-1], values[::-1], color=palette[0], height=0.65)

# Подписи значений у столбцов
for bar, val in zip(bars, values[::-1]):
    ax.text(
        bar.get_width() + 0.3, bar.get_y() + bar.get_height() / 2,
        str(val), va="center", ha="left", fontsize=10, color=VIZ["node_text"]
    )

ax.set_title("Распределение частей речи в корпусе")
ax.set_xlabel("Количество токенов")
ax.set_xlim(0, max(values) * 1.15)
ax.grid(axis="x")
ax.grid(axis="y", linewidth=0)
fig.tight_layout()
plt.show()

### Шаг 4.4. Именованные сущности (NER)

Ключевая задача для научных текстов — автоматическое извлечение именованных сущностей: названий организаций, имён учёных, географических объектов, дат и числовых значений. spaCy выполняет NER в рамках того же пайплайна и возвращает список объектов `ent` с атрибутами `text`, `label_` и позицией в тексте.

`displacy.render()` создаёт HTML-визуализацию с цветовой маркировкой сущностей прямо в Colab.

In [ ]:
from spacy import displacy
from IPython.display import HTML, display as ipy_display

# Метки NER в ru_core_news_sm и их значение
NER_LABELS_RU = {
    "PER":  "Персона",
    "ORG":  "Организация",
    "LOC":  "Место",
    "DATE": "Дата",
    "TIME": "Время",
    "MONEY": "Сумма",
    "PERCENT": "Процент",
    "MISC": "Прочее",
}

# Показываем NER для первых четырёх предложений корпуса
demo_text = " ".join(CORPUS[:4])
doc_ner = nlp_ru(demo_text)

# displacy.render возвращает HTML-строку; jupyter=True включает
# встроенный стиль без внешних ресурсов.
html = displacy.render(doc_ner, style="ent", jupyter=False,
                       options={"compact": True})
ipy_display(HTML(html))

print()
print("Найденные сущности:")
print(f"{'Текст':<35} {'Метка':<10} {'Значение'}")
print("-" * 60)
for ent in doc_ner.ents:
    label_ru = NER_LABELS_RU.get(ent.label_, ent.label_)
    print(f"{ent.text:<35} {ent.label_:<10} {label_ru}")

### Шаг 4.5. Статистика именованных сущностей по всему корпусу

Посчитаем, сколько сущностей каждого типа встречается в корпусе, и визуализируем распределение.

In [ ]:
import matplotlib.pyplot as plt

# Собираем все сущности из полного корпуса
doc_full = nlp_ru(FULL_TEXT)
ent_type_counts = Counter(ent.label_ for ent in doc_full.ents)
ent_text_counts = Counter(ent.text for ent in doc_full.ents)

print(f"Всего именованных сущностей в корпусе: {len(doc_full.ents)}")
print(f"Уникальных сущностей: {len(set(ent.text for ent in doc_full.ents))}")
print()

# Топ-10 конкретных сущностей по встречаемости
print("Наиболее частые именованные сущности:")
for text, cnt in ent_text_counts.most_common(10):
    print(f"  {text:<35} {cnt}")

# График: распределение типов сущностей
type_items = sorted(ent_type_counts.items(), key=lambda x: -x[1])
type_labels = [NER_LABELS_RU.get(t, t) for t, _ in type_items]
type_values = [c for _, c in type_items]

fig, ax = plt.subplots(figsize=(9, 4.5))
bars = ax.bar(
    range(len(type_labels)), type_values,
    color=[VIZ["series"][i % 4] for i in range(len(type_labels))],
    width=0.6,
)
ax.set_xticks(range(len(type_labels)))
ax.set_xticklabels(type_labels, rotation=20, ha="right", fontsize=10)

# Подписи над столбцами
for bar, val in zip(bars, type_values):
    ax.text(
        bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.1,
        str(val), ha="center", va="bottom", fontsize=10, color=VIZ["node_text"]
    )

ax.set_title("Типы именованных сущностей в корпусе")
ax.set_ylabel("Количество упоминаний")
ax.set_ylim(0, max(type_values) * 1.2)
ax.grid(axis="x", linewidth=0)
fig.tight_layout()
plt.show()

### Шаг 4.6. Синтаксические зависимости

Парсер зависимостей определяет грамматическую структуру предложения: какое слово является главным, а какое — зависимым, и какова природа связи (подлежащее, прямое дополнение, определение и т.д.). `displacy` рисует дерево зависимостей как SVG-схему прямо в Colab.

In [ ]:
from spacy import displacy
from IPython.display import HTML, display as ipy_display

# Берём короткое предложение для наглядного дерева
short_sent = "Профессор Мария Соколова опубликовала монографию о роли микробиома."
doc_dep = nlp_ru(short_sent)

# Рендерим дерево синтаксических зависимостей
svg = displacy.render(
    doc_dep,
    style="dep",
    jupyter=False,
    options={
        "compact":  False,
        "distance": 110,
        "arrow_stroke": 2,
        "word_spacing": 45,
    }
)
ipy_display(HTML(svg))

# Программный доступ к зависимостям (для дальнейшей обработки)
print("\nСинтаксические связи:")
print(f"{'Токен':<20} {'Голова':<20} {'Зависимость'}")
print("-" * 55)
for token in doc_dep:
    print(f"{token.text:<20} {token.head.text:<20} {token.dep_}")

### Шаг 4.7. spaCy как препроцессинг перед ML

Один из самых практичных сценариев для учёного: использовать spaCy для нормализации текста перед передачей в классификатор или кластеризатор. Покажем полный мини-пайплайн:

1. Лемматизируем и фильтруем токены с помощью spaCy.
2. Строим TF-IDF-матрицу признаков из лемм (scikit-learn).
3. Обучаем простой классификатор, который определяет, к какой тематической области относится предложение.

Это иллюстрирует роль spaCy как первого этапа в полноценном NLP-пайплайне машинного обучения.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

# Размечаем предложения по тематике вручную (игрушечный датасет)
# 0 — биология/медицина, 1 — физика/химия, 2 — организационные/статистические
LABELS = [0, 1, 1, 2, 0, 2, 0, 1, 1, 2]
TOPIC_NAMES = {0: "Биология / медицина", 1: "Физика / химия", 2: "Орг. / статистика"}

# Шаг 1: лемматизация через spaCy
# Каждое предложение → строка из значимых лемм через пробел
def lemmatize_sentence(text, nlp):
    """Преобразует предложение в строку лемм (без стоп-слов и пунктуации)."""
    doc = nlp(text)
    lemmas = [
        token.lemma_.lower()
        for token in doc
        if not token.is_punct and not token.is_space and not token.is_stop
        and len(token.lemma_) > 2
    ]
    return " ".join(lemmas)

lemmatized = [lemmatize_sentence(sent, nlp_ru) for sent in CORPUS]

print("Пример лемматизации:")
for orig, lem in zip(CORPUS[:3], lemmatized[:3]):
    print(f"  ИСХОДНЫЙ: {orig[:70]}...")
    print(f"  ЛЕММЫ:    {lem}")
    print()

# Шаг 2: TF-IDF по леммам
vectorizer = TfidfVectorizer(min_df=1)
X_tfidf = vectorizer.fit_transform(lemmatized)
y = np.array(LABELS)

print(f"Размер TF-IDF матрицы: {X_tfidf.shape}  (предложений × уникальных лемм)")
print()

# Шаг 3: обучение классификатора
# При выборке в 10 документов используем LOO-кросс-валидацию
from sklearn.model_selection import LeaveOneOut
clf = LogisticRegression(max_iter=1000, random_state=42, C=1.0)
loo = LeaveOneOut()
loo_scores = cross_val_score(clf, X_tfidf, y, cv=loo, scoring="accuracy")

print(f"Точность классификации (LOO cross-val): {loo_scores.mean():.2f}")
print(f"Правильно классифицировано: {int(loo_scores.sum())} из {len(CORPUS)}")
print()
print("Примечание: выборка из 10 предложений слишком мала для реального")
print("исследования — здесь важна демонстрация цикла, а не абсолютная точность.")

# Шаг 4: важность TF-IDF признаков — какие леммы наиболее характерны
clf.fit(X_tfidf, y)
feature_names = vectorizer.get_feature_names_out()

fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))
for class_idx, ax in enumerate(axes):
    # Коэффициенты для данного класса
    coefs = clf.coef_[class_idx]
    top_idx = np.argsort(coefs)[-8:]
    ax.barh(
        feature_names[top_idx],
        coefs[top_idx],
        color=VIZ["series"][class_idx % 4],
        height=0.65,
    )
    ax.set_title(TOPIC_NAMES[class_idx], fontsize=11)
    ax.set_xlabel("Вес признака")
    ax.tick_params(axis="y", labelsize=9)

fig.suptitle("Ключевые леммы по тематическим классам (TF-IDF + LogReg)",
             fontsize=13, fontweight="semibold", y=1.02)
fig.tight_layout()
plt.show()

## 5. Интерпретация результатов

**Токенизация и лемматизация.** Таблица из шага 4.1 показывает, как spaCy понимает каждое слово: `POS` — часть речи в универсальной схеме UD (NOUN, VERB, ADJ и т.д.), `TAG` — более детальный морфологический тег с грамматическими категориями (род, число, падеж). Лемма убирает флексию: «исследований» → «исследование», что позволяет корректно сравнивать разные формы одного слова.

**Распределение частей речи.** В научном тексте преобладают существительные (NOUN) и имена собственные (PROPN) — это следствие терминологической насыщенности. Высокая доля прилагательных (ADJ) характерна для описательных разделов. Если ваш корпус даёт неожиданный профиль POS, это может указывать на проблемы кодировки или смешение языков.

**Именованные сущности.** displacy-визуализация окрашивает спаны разными цветами по типу: ORG — организации, PER — персоны, LOC — места, DATE — даты. Модель `ru_core_news_sm` обучена на новостных текстах, поэтому в строго научных текстах (с аббревиатурами, химическими формулами) могут быть пропуски или ошибки — это нормально для малой модели. Для точного NER в специализированных доменах (биомедицина, юриспруденция) потребуется дообучение модели.

**Синтаксические зависимости.** В дереве зависимостей (шаг 4.6) корень (`ROOT`) — это обычно глагол-сказуемое. Стрелки показывают направление зависимости. Метки типа `nsubj` (подлежащее), `dobj` (прямое дополнение), `amod` (согласованное определение) позволяют программно извлекать факты вида «кто — что сделал — с чем».

**ML-пайплайн.** График важности признаков (шаг 4.7) показывает, какие леммы наиболее информативны для каждого класса. Лемматизация через spaCy сокращает словарь TF-IDF матрицы на 20–40% по сравнению с сырыми словоформами, что улучшает обобщение на малых корпусах. На реальных данных (1000+ документов) этот эффект заметен особенно ярко.

## 6. Попробуйте на своих данных

Вставьте собственные тексты в переменную `MY_TEXTS` ниже. Ячейка автоматически выполнит:
- токенизацию и лемматизацию;
- извлечение именованных сущностей с displacy-подсветкой;
- подсчёт частот POS и сущностей.

Требования: тексты должны быть строками Python на русском языке. Для загрузки из файла используйте `open()` или `pd.read_csv()`.

In [ ]:
# --- Шаблон для своих данных ---
# Вставьте ваши тексты в список MY_TEXTS.
# Каждый элемент — одно предложение или абзац.

MY_TEXTS = [
    "Вставьте сюда первый абзац вашего текста.",
    "Вставьте сюда второй абзац вашего текста.",
    # "Третий абзац...",
]

# ------------------------------------------------------------------
# Всё ниже выполняется автоматически — редактировать не нужно.
# ------------------------------------------------------------------

from spacy import displacy
from IPython.display import HTML, display as ipy_display
from collections import Counter
import matplotlib.pyplot as plt

# Убедимся, что модель загружена (если пропустили шаг 4.1)
try:
    nlp_ru
except NameError:
    import spacy
    nlp_ru = spacy.load("ru_core_news_sm")

my_full_text = " ".join(MY_TEXTS)
my_doc = nlp_ru(my_full_text)

# Лемматизация
my_lemmas = [
    token.lemma_.lower()
    for token in my_doc
    if not token.is_punct and not token.is_space and not token.is_stop
]
print(f"Токенов: {len(my_doc)}  |  Значимых лемм: {len(my_lemmas)}")
print(f"Топ-10 лемм: {Counter(my_lemmas).most_common(10)}")
print()

# NER с displacy
print("Именованные сущности:")
html = displacy.render(my_doc, style="ent", jupyter=False,
                       options={"compact": True})
ipy_display(HTML(html))

# Найденные сущности списком
if my_doc.ents:
    print("\nСписок сущностей:")
    for ent in my_doc.ents:
        label_ru = NER_LABELS_RU.get(ent.label_, ent.label_)
        print(f"  {ent.text:<35} {label_ru}")
else:
    print("Именованных сущностей не найдено. "
          "Проверьте, что текст написан на русском языке.")

# POS-профиль
my_pos = Counter(
    token.pos_ for token in my_doc
    if not token.is_punct and not token.is_space
)
if my_pos:
    items = my_pos.most_common()
    labels_plot = [POS_LABELS.get(p, p) for p, _ in items]
    values_plot = [c for _, c in items]
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.bar(range(len(labels_plot)), values_plot,
           color=VIZ["series"][0], width=0.6)
    ax.set_xticks(range(len(labels_plot)))
    ax.set_xticklabels(labels_plot, rotation=20, ha="right", fontsize=10)
    ax.set_title("Распределение частей речи в вашем тексте")
    ax.set_ylabel("Количество токенов")
    fig.tight_layout()
    plt.show()

## Что дальше

- Официальная документация spaCy: https://spacy.io/
- Список всех русскоязычных моделей (включая среднюю `ru_core_news_md` и большую `ru_core_news_lg`): https://spacy.io/models/ru
- Руководство по обучению собственной NER-модели на доменных данных: https://spacy.io/usage/training
- Компонент `spacy-transformers` для подключения BERT-подобных моделей: https://spacy.io/usage/embeddings-transformers
- Библиотека `spaCy Projects` — шаблоны воспроизводимых NLP-пайплайнов: https://spacy.io/usage/projects
- Для биомедицинского NER рассмотрите модели серии `en_core_sci_*` из пакета `scispaCy` (Allen AI): https://allenai.github.io/scispacy/

## Три вопроса для самопроверки

Ответьте до того, как раскроете подсказку, — это проверяет, что метод понят, а не просто запущен.

1. NER-компонент модели `ru_core_news_sm` пропустил большинство названий химических соединений и аббревиатур реагентов в вашем корпусе лабораторных протоколов. Чем это объясняется и какой практический шаг позволит улучшить качество распознавания именно на таких текстах?
2. Вы лемматизировали корпус и построили TF-IDF-матрицу; LOO-точность классификатора составила 0.50 при трёх классах. Вы предполагаете, что предобработка некорректна. Какой конкретный артефакт лемматизации мог снизить качество, если корпус состоит из узкоспециальных терминов, отсутствующих в обучающих данных модели?
3. Два текста имеют одинаковый POS-профиль (распределение частей речи) и набор частотных лемм, но относятся к разным предметным областям. Какой инструмент из пайплайна spaCy позволяет извлечь структурированные отношения между сущностями внутри предложения, а не только саму частоту слов?

<details>
<summary>Показать ориентиры для ответов</summary>

1. Модель `ru_core_news_sm` обучена на новостных текстах: она не видела химических аббревиатур, торговых названий реагентов и нотации типа «SnCl₂» или «ДМСО». Это доменный сдвиг. Практический путь: аннотировать несколько сотен предложений из целевого корпуса и дообучить NER-компонент через `spacy train`, либо добавить компонент на основе правил (`EntityRuler`) с паттернами для известных аббревиатур.
2. Если термин не встречался в обучающих данных модели, spaCy может вернуть неверную лемму или вовсе оставить словоформу без изменений. Более серьёзный артефакт: модель может ошибочно определить часть речи специального термина (например, принять латинскую аббревиатуру за имя собственное), что повлечёт неверную лемму и смешение двух разных понятий под одной формой в TF-IDF словаре.
3. Парсер синтаксических зависимостей (`dep`) строит дерево связей: из него можно программно извлекать тройки «субъект — предикат — объект» или пары «существительное — его определение». В сочетании с NER это позволяет получать структурированные факты вида «организация — провела — эксперимент» вместо простой частотной статистики слов.
</details>